<a href="https://colab.research.google.com/github/leticiavitoriadev/ATAL/blob/main/Problema_de_backtracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Sudoku: resolve um Sudoku pequeno e imprime a solução.

N é a ordem do bloco: o tabuleiro tem N² x N² casas e blocos N x N
(N=2 -> 4x4, N=3 -> 9x9 clássico, N=4 -> 16x16, N=5 -> 25x25).

Entrada: N, seguido das N² linhas do tabuleiro, com os valores separados
por espaço e 0 (ou '.') nas casas vazias. Pode vir de um arquivo
(python sudoku.py arquivo.txt) ou ser digitada. Sem entrada, roda os
exemplos embutidos.

Busca: backtracking com máscaras de bits para linhas, colunas e blocos.
Em cada passo ramifica pela opção mais restrita: a casa vazia com menos
candidatos (heurística MRV) ou o valor que cabe em menos casas de alguma
linha/coluna/bloco ("único lugar"). Isso torna viável até o 25x25.
"""
import sys


def resolver(n, grade):
    t = n * n
    cheio = (1 << t) - 1
    lin = [0] * t
    col = [0] * t
    blo = [0] * t
    vazias = []

    def b(i, j):
        return (i // n) * n + j // n

    for i in range(t):
        for j in range(t):
            v = grade[i][j]
            if v == 0:
                vazias.append((i, j))
                continue
            bit = 1 << (v - 1)
            if not 1 <= v <= t or (lin[i] | col[j] | blo[b(i, j)]) & bit:
                return None                      # tabuleiro inicial inválido
            lin[i] |= bit; col[j] |= bit; blo[b(i, j)] |= bit

    g = [row[:] for row in grade]
    unidades = ([[(i, j) for j in range(t)] for i in range(t)] +
                [[(i, j) for i in range(t)] for j in range(t)] +
                [[(bi * n + a, bj * n + c) for a in range(n) for c in range(n)]
                 for bi in range(n) for bj in range(n)])

    def bt():
        if not vazias:
            return True
        # candidatos de cada casa vazia
        cands = {}
        melhor_k, escolha = t + 1, None          # escolha = lista de (i, j, bit)
        for (i, j) in vazias:
            c = cheio & ~(lin[i] | col[j] | blo[b(i, j)])
            if c == 0:
                return False                     # casa sem candidato
            cands[(i, j)] = c
            k = bin(c).count("1")
            if k < melhor_k:
                melhor_k = k
                escolha = [(i, j, 1 << q) for q in range(t) if c >> q & 1]
        # "único lugar": um valor que falta numa linha/coluna/bloco e só
        # cabe em poucas casas dela também serve para ramificar
        if melhor_k > 1:
            for u in unidades:
                presentes = 0
                lugares = {}                     # bit -> casas onde cabe
                for (i, j) in u:
                    v = g[i][j]
                    if v:
                        presentes |= 1 << (v - 1)
                        continue
                    c = cands[(i, j)]
                    while c:
                        bit = c & -c
                        c ^= bit
                        lugares.setdefault(bit, []).append((i, j))
                faltam = cheio & ~presentes
                if bin(faltam).count("1") != len(lugares):
                    return False                 # valor sem lugar possível
                for bit, casas in lugares.items():
                    if len(casas) < melhor_k:
                        melhor_k = len(casas)
                        escolha = [(i, j, bit) for (i, j) in casas]
                        if melhor_k == 1:
                            break
                if melhor_k == 1:
                    break
        for (i, j, bit) in escolha:
            idx = vazias.index((i, j))
            vazias[idx] = vazias[-1]; vazias.pop()
            bb = b(i, j)
            g[i][j] = bit.bit_length()
            lin[i] |= bit; col[j] |= bit; blo[bb] |= bit
            if bt():
                return True
            lin[i] ^= bit; col[j] ^= bit; blo[bb] ^= bit
            g[i][j] = 0
            vazias.append((i, j))
        return False

    return g if bt() else None


def imprimir(n, g):
    t = n * n
    larg = len(str(t))
    sep = "+".join(["-" * ((larg + 1) * n + 1)] * n)
    for i in range(t):
        if i and i % n == 0:
            print(sep)
        partes = []
        for k in range(n):
            partes.append(" " + " ".join(str(v).rjust(larg) for v in g[i][k * n:(k + 1) * n]) + " ")
        print("|".join(partes))


def ler(texto):
    tokens = texto.replace(".", " 0 ").split()
    n = int(tokens[0])
    t = n * n
    vals = [int(x) for x in tokens[1:1 + t * t]]
    if len(vals) != t * t:
        raise ValueError(f"esperados {t*t} valores para N={n}, lidos {len(vals)}")
    return n, [vals[k * t:(k + 1) * t] for k in range(t)]


EXEMPLOS = [
    """2
    1 0 0 0
    0 0 3 0
    0 4 0 0
    0 0 0 2""",
    """3
    5 3 0 0 7 0 0 0 0
    6 0 0 1 9 5 0 0 0
    0 9 8 0 0 0 0 6 0
    8 0 0 0 6 0 0 0 3
    4 0 0 8 0 3 0 0 1
    7 0 0 0 2 0 0 0 6
    0 6 0 0 0 0 2 8 0
    0 0 0 4 1 9 0 0 5
    0 0 0 0 8 0 0 7 9""",
]


def resolver_texto(texto):
    n, grade = ler(texto)
    print(f"Problema (N={n}, tabuleiro {n*n}x{n*n}):")
    imprimir(n, grade)
    sol = resolver(n, grade)
    print()
    if sol is None:
        print("Não há solução.")
    else:
        print("Solução:")
        imprimir(n, sol)
    print()


def main():
    if len(sys.argv) == 2:
        with open(sys.argv[1], encoding="utf-8") as f:
            resolver_texto(f.read())
    else:
        texto = "" if sys.stdin.isatty() else sys.stdin.read()
        if texto.strip():
            resolver_texto(texto)
            return
        print("Sem entrada: resolvendo os exemplos embutidos.\n")
        for ex in EXEMPLOS:
            resolver_texto(ex)


if __name__ == "__main__":
    main()


Sem entrada: resolvendo os exemplos embutidos.

Problema (N=2, tabuleiro 4x4):
 1 0 | 0 0 
 0 0 | 3 0 
-----+-----
 0 4 | 0 0 
 0 0 | 0 2 

Solução:
 1 3 | 2 4 
 4 2 | 3 1 
-----+-----
 2 4 | 1 3 
 3 1 | 4 2 

Problema (N=3, tabuleiro 9x9):
 5 3 0 | 0 7 0 | 0 0 0 
 6 0 0 | 1 9 5 | 0 0 0 
 0 9 8 | 0 0 0 | 0 6 0 
-------+-------+-------
 8 0 0 | 0 6 0 | 0 0 3 
 4 0 0 | 8 0 3 | 0 0 1 
 7 0 0 | 0 2 0 | 0 0 6 
-------+-------+-------
 0 6 0 | 0 0 0 | 2 8 0 
 0 0 0 | 4 1 9 | 0 0 5 
 0 0 0 | 0 8 0 | 0 7 9 

Solução:
 5 3 4 | 6 7 8 | 9 1 2 
 6 7 2 | 1 9 5 | 3 4 8 
 1 9 8 | 3 4 2 | 5 6 7 
-------+-------+-------
 8 5 9 | 7 6 1 | 4 2 3 
 4 2 6 | 8 5 3 | 7 9 1 
 7 1 3 | 9 2 4 | 8 5 6 
-------+-------+-------
 9 6 1 | 5 3 7 | 2 8 4 
 2 8 7 | 4 1 9 | 6 3 5 
 3 4 5 | 2 8 6 | 1 7 9 

